# Adaptive Hybrid RecSys — Data Pipeline
**Порядок:** Runtime → Run all

Память оптимизирована: TF-IDF хранится sparse (~120 MB вместо ~2 GB).

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR     = '/content/drive/MyDrive/disser'
RAW_DIR       = f'{DRIVE_DIR}/data/raw'
PROCESSED_DIR = f'{DRIVE_DIR}/data/processed'
RECBOLE_DIR   = f'{DRIVE_DIR}/data/recbole/amazon-electronics'

for d in [RAW_DIR, PROCESSED_DIR, RECBOLE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Drive OK. Processed dir: {PROCESSED_DIR}')

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
!pip install -q recbole pandas scikit-learn pyarrow PyYAML tqdm
print('Done')

In [ ]:
# ── 3. Check RAM before start ──────────────────────────────────────────────
import psutil, gc

def ram_gb():
    m = psutil.virtual_memory()
    return m.used/1e9, m.available/1e9, m.total/1e9

def free_ram():
    gc.collect()
    used, avail, total = ram_gb()
    print(f'RAM: {used:.1f} GB used / {total:.1f} GB total ({avail:.1f} GB free)')

free_ram()

In [ ]:
# ── 4. Download data ───────────────────────────────────────────────────────
import urllib.request
from pathlib import Path

REVIEWS_URL  = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/categoryFilesSmall/Electronics_5.json.gz'
METADATA_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/metaFiles2/meta_Electronics.json.gz'

def download(url, dest):
    p = Path(dest)
    if p.exists() and p.stat().st_size > 1e6:
        print(f'Already exists: {p.name} ({p.stat().st_size/1e9:.2f} GB)')
        return
    print(f'Downloading {p.name}...')
    urllib.request.urlretrieve(url, dest)
    print(f'Done: {p.stat().st_size/1e9:.2f} GB')

download(REVIEWS_URL,  f'{RAW_DIR}/Electronics_5.json.gz')
download(METADATA_URL, f'{RAW_DIR}/meta_Electronics.json.gz')

In [ ]:
# ── 5. Load & filter reviews ───────────────────────────────────────────────
# Загружаем только нужные колонки, сразу освобождаем лишнее
import json, gzip
import pandas as pd
import numpy as np

print('Loading reviews (only needed columns)...')
records = []
with gzip.open(f'{RAW_DIR}/Electronics_5.json.gz', 'rt', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        try:
            rec = json.loads(line)
            records.append((
                rec.get('reviewerID', ''),
                rec.get('asin', ''),
                float(rec.get('overall', 0)),
                int(rec.get('unixReviewTime', 0)),
            ))
        except Exception:
            continue
        if (i+1) % 2_000_000 == 0:
            print(f'  {i+1:,} lines...')

reviews = pd.DataFrame(records, columns=['user_id','item_id','rating','timestamp'])
del records
gc.collect()
print(f'Loaded: {len(reviews):,} reviews')
free_ram()

In [ ]:
# ── 6. k-core filtering ────────────────────────────────────────────────────
def kcore(df, k=5):
    prev = -1
    while len(df) != prev:
        prev = len(df)
        vc = df['user_id'].value_counts()
        df = df[df['user_id'].isin(vc[vc >= k].index)]
        vc = df['item_id'].value_counts()
        df = df[df['item_id'].isin(vc[vc >= k].index)]
    return df.reset_index(drop=True)

interactions = kcore(reviews, k=5)
del reviews
gc.collect()

# ID maps (1-based, 0 = padding)
user_map = {u: i+1 for i,u in enumerate(sorted(interactions['user_id'].unique()))}
item_map = {it: i+1 for i,it in enumerate(sorted(interactions['item_id'].unique()))}
interactions['user_idx'] = interactions['user_id'].map(user_map).astype('int32')
interactions['item_idx'] = interactions['item_id'].map(item_map).astype('int32')
interactions['rating']   = interactions['rating'].astype('float32')
interactions = interactions.sort_values('timestamp').reset_index(drop=True)

n_users, n_items = len(user_map), len(item_map)
print(f'After filter: {len(interactions):,} interactions | {n_users:,} users | {n_items:,} items')
free_ram()

In [ ]:
# ── 7. Temporal split ──────────────────────────────────────────────────────
n = len(interactions)
train_df = interactions.iloc[:int(n*0.70)].copy()
val_df   = interactions.iloc[int(n*0.70):int(n*0.85)].copy()
test_df  = interactions.iloc[int(n*0.85):].copy()

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
free_ram()

In [ ]:
# ── 8. Temporal features (лёгкие) ─────────────────────────────────────────
def build_temporal(df):
    dt  = pd.to_datetime(df['timestamp'], unit='s')
    hour = dt.dt.hour.values
    dow  = dt.dt.dayofweek.values
    return pd.DataFrame({
        'hour_sin':  np.sin(2*np.pi*hour/24).astype('float32'),
        'hour_cos':  np.cos(2*np.pi*hour/24).astype('float32'),
        'dow_sin':   np.sin(2*np.pi*dow/7).astype('float32'),
        'dow_cos':   np.cos(2*np.pi*dow/7).astype('float32'),
        'is_weekend':(dow >= 5).astype('float32'),
    }, index=df.index)

train_temporal = build_temporal(train_df)
val_temporal   = build_temporal(val_df)
test_temporal  = build_temporal(test_df)
print(f'Temporal features: {train_temporal.shape}')
free_ram()

In [ ]:
# ── 9. User sequences ─────────────────────────────────────────────────────
MAX_SEQ = 50
user_sequences = {}
for uid, grp in interactions.sort_values('timestamp').groupby('user_idx'):
    items = grp['item_idx'].values
    user_sequences[int(uid)] = items[-MAX_SEQ:].astype(np.int32)

print(f'Sequences: {len(user_sequences):,} users')
lens = [len(v) for v in user_sequences.values()]
print(f'  avg_len={np.mean(lens):.1f}, max_len={max(lens)}')
free_ram()

In [ ]:
# ── 10. TF-IDF item features (SPARSE — экономия RAM x10) ──────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

print('Loading metadata...')
meta_records = []
with gzip.open(f'{RAW_DIR}/meta_Electronics.json.gz', 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        try:
            rec = json.loads(line)
            cats = rec.get('category', [])
            if isinstance(cats, list):
                cats = [c for sub in cats for c in (sub if isinstance(sub, list) else [sub])]
            meta_records.append({
                'item_id': rec.get('asin',''),
                'title':   rec.get('title',''),
                'category': ' '.join(cats),
            })
        except Exception:
            continue

metadata = pd.DataFrame(meta_records).drop_duplicates('item_id')
del meta_records
gc.collect()

# Оставляем только items из interactions
valid_items = set(interactions['item_id'].unique())
metadata = metadata[metadata['item_id'].isin(valid_items)].copy()
metadata['item_idx'] = metadata['item_id'].map(item_map).astype('int32')
metadata['text'] = metadata['title'].fillna('') + ' ' + metadata['category'].fillna('')
print(f'Metadata: {len(metadata):,} items')
free_ram()

# TF-IDF — оставляем SPARSE!
print('Building TF-IDF (sparse)...')
vec = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2, max_df=0.95)
tfidf_sparse = vec.fit_transform(metadata['text'])  # shape: (n_items_in_meta, 5000)

# Строим sparse матрицу (max_item_idx+1) x 5000, idx→row
max_idx = int(metadata['item_idx'].max())
rows, cols, vals = [], [], []
cx = tfidf_sparse.tocoo()
item_idxs = metadata['item_idx'].values
for r, c, v in zip(cx.row, cx.col, cx.data):
    real_idx = item_idxs[r]
    rows.append(real_idx)
    cols.append(c)
    vals.append(v)

item_content_sparse = sp.csr_matrix(
    (vals, (rows, cols)),
    shape=(max_idx + 1, tfidf_sparse.shape[1]),
    dtype=np.float32
)
del tfidf_sparse, cx, rows, cols, vals, metadata
gc.collect()

content_dim = item_content_sparse.shape[1]
print(f'TF-IDF sparse: {item_content_sparse.shape}, nnz={item_content_sparse.nnz:,}')
print(f'  Dense would be: {item_content_sparse.shape[0]*item_content_sparse.shape[1]*4/1e9:.2f} GB')
print(f'  Sparse is:      {item_content_sparse.data.nbytes/1e6:.1f} MB')
free_ram()

In [ ]:
# ── 11. Save everything to Drive ──────────────────────────────────────────
import pickle, json
from pathlib import Path
P = Path(PROCESSED_DIR)

print('Saving...')

# DataFrames
interactions.to_parquet(P/'interactions.parquet', index=False)
train_df.to_parquet(P/'train.parquet', index=False)
val_df.to_parquet(P/'val.parquet', index=False)
test_df.to_parquet(P/'test.parquet', index=False)
train_temporal.to_parquet(P/'train_temporal.parquet', index=False)
val_temporal.to_parquet(P/'val_temporal.parquet', index=False)
test_temporal.to_parquet(P/'test_temporal.parquet', index=False)

# Sparse TF-IDF (scipy format, ~100MB вместо 2GB)
sp.save_npz(str(P/'item_content_sparse.npz'), item_content_sparse)

# User sequences
np.savez(str(P/'user_sequences.npz'), sequences=user_sequences)

# ID maps
with open(P/'user_map.pkl','wb') as f: pickle.dump(user_map, f)
with open(P/'item_map.pkl','wb') as f: pickle.dump(item_map, f)

# Stats
stats = {
    'n_users': n_users, 'n_items': n_items,
    'n_interactions': len(interactions),
    'train_size': len(train_df), 'val_size': len(val_df), 'test_size': len(test_df),
    'content_dim': content_dim, 'context_dim': 5,
    'tfidf_sparse': True,
}
with open(P/'dataset_stats.json','w') as f: json.dump(stats, f, indent=2)

print('\nSaved files:')
for f in sorted(P.glob('*')):
    print(f'  {f.name:45s} {f.stat().st_size/1e6:8.1f} MB')

In [ ]:
# ── 12. RecBole .inter file (для baseline'ов) ──────────────────────────────
inter_path = Path(RECBOLE_DIR) / 'amazon-electronics.inter'
if not inter_path.exists():
    print('Creating RecBole .inter file...')
    with open(inter_path, 'w') as f:
        f.write('user_id:token\titem_id:token\trating:float\ttimestamp:float\n')
        for row in interactions[['user_id','item_id','rating','timestamp']].itertuples(index=False):
            f.write(f'{row.user_id}\t{row.item_id}\t{row.rating}\t{row.timestamp}\n')
    print(f'Done: {inter_path.stat().st_size/1e6:.1f} MB')
else:
    print(f'Already exists: {inter_path.name}')

In [ ]:
# ── 13. Verify (без загрузки в RAM, только статистика) ────────────────────
import json
from pathlib import Path
import scipy.sparse as sp
import numpy as np

P = Path(PROCESSED_DIR)
stats = json.load(open(P/'dataset_stats.json'))

print('=== DATASET STATISTICS ===')
for k, v in stats.items():
    print(f'  {k:<20}: {v:,}' if isinstance(v, int) else f'  {k:<20}: {v}')

print()
print('Files on Drive:')
total_mb = 0
for f in sorted(P.glob('*')):
    mb = f.stat().st_size/1e6
    total_mb += mb
    print(f'  {f.name:45s} {mb:8.1f} MB')
print(f'  {"TOTAL":45s} {total_mb:8.1f} MB')

free_ram()
print('\n✓ Data pipeline complete! Download data/processed/ to local machine.')